# DS-Fall - 01 Process and Visualize Data

This notebook converts WEDA-FALL, HIFD/HR_IMU, and BITS-2/Dataset into one unified format: `(N, 100, 6)` at 50 Hz, 2 seconds per window, channel order `[ax, ay, az, gx, gy, gz]`.

BITS-2 is always resampled from 20 Hz to 50 Hz using row-order uniform interpolation before windowing.

In [ ]:
# Colab setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/ds-fall')
# For local debugging, uncomment and adjust:
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQ = PROJECT_ROOT / 'requirements.txt'
if REQ.exists():
    print('Project root:', PROJECT_ROOT)
else:
    print('WARNING: requirements.txt not found. Check PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# Install requirements only when core packages are missing.
import importlib
import subprocess
MODULE_CHECKS = [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'), ('tensorflow', 'tensorflow')]
def _module_ok(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False
missing = [pkg for module, pkg in MODULE_CHECKS if not _module_ok(module)]
if missing and REQ.exists():
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)])

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import make_config
from src.utils.io import ensure_dir
from src.utils.seed import set_seed
from src.data.preprocessing import (
    build_all_windows,
    build_preprocessing_summary,
    normalize_with_train_scaler,
    save_processed_dataset,
)
from src.data.splitting import subject_wise_split
from src.data.visualization import (
    plot_bits_resampling,
    plot_channel_stats,
    plot_class_distribution,
    plot_dataset_distribution,
    plot_imu_window,
)

set_seed(42)
config = make_config(PROJECT_ROOT)
for path in [config.processed_dir, config.figures_dir / 'preprocessing', config.models_dir, config.logs_dir, config.metrics_dir]:
    ensure_dir(path)

RAW_DIR = config.raw_dir
PROCESSED_DIR = config.processed_dir
OUTPUT_DIR = config.output_dir
BITS_RAW_DIR = config.bits_raw_dir
HIFD_RAW_DIR = config.hifd_raw_dir
WEDA_RAW_DIR = config.weda_raw_dir

print('PROJECT_ROOT =', config.project_root)
print('RAW_DIR =', RAW_DIR)
for name, path in [('WEDA', WEDA_RAW_DIR), ('HIFD', HIFD_RAW_DIR), ('BITS', BITS_RAW_DIR)]:
    print(f'{name}:', path, 'exists=' + str(path.exists()))

## Build windows

The loader skips missing dataset folders with warnings and continues with all available datasets.

In [ ]:
X_raw, metadata, summary = build_all_windows(config)
print('Raw window shape:', X_raw.shape)
if len(X_raw) == 0:
    raise RuntimeError('No windows were created. Check raw dataset placement under data/raw.')

display(metadata.head())
print(metadata[['dataset', 'fall_label', 'direction_label']].value_counts().head(20))

## Subject-wise split and train-only normalization

Normalization is fit only on train subjects, then applied to train/val/test.

In [ ]:
metadata, split_subjects = subject_wise_split(metadata, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=config.seed)

print('Split subjects:')
print(json.dumps(split_subjects, indent=2))
print('\nWindows by split:')
print(metadata['split'].value_counts())

X, scaler, norm_stats = normalize_with_train_scaler(X_raw, metadata)
summary.update(build_preprocessing_summary(X, metadata))
summary['normalization'] = norm_stats
summary['bits_preprocessing'] = '20Hz row-order uniform interpolation to 50Hz before 2-second windowing'

print('Normalized X shape:', X.shape)
print('NaN count:', np.isnan(X).sum(), 'Inf count:', np.isinf(X).sum())
print('Scaler:', scaler)

## Save processed dataset

In [ ]:
save_processed_dataset(PROCESSED_DIR, X, metadata, scaler, split_subjects, summary)
print('Saved processed files to:', PROCESSED_DIR)
print(sorted([p.name for p in PROCESSED_DIR.iterdir() if p.is_file()]))

## Raw and resampling visual checks

In [ ]:
fig_dir = config.figures_dir / 'preprocessing'

# Raw-like examples loaded before normalization when possible.
from src.data.weda_loader import find_weda_trials, load_weda_trial
from src.data.hifd_loader import find_hifd_trials, load_hifd_mat
from src.data.bits_loader import find_bits_trials, parse_bits_csv, resample_bits_sequence_to_50hz

if WEDA_RAW_DIR.exists():
    weda_trials = find_weda_trials(WEDA_RAW_DIR)
    for label, predicate in [('weda_fall_raw', lambda t: t['activity_id'].startswith('F')), ('weda_adl_raw', lambda t: t['activity_id'].startswith('D'))]:
        trial = next((t for t in weda_trials if predicate(t)), None)
        if trial:
            seq = load_weda_trial(trial['accel_path'], trial['gyro_path'])
            if seq is not None and len(seq) >= 100:
                plot_imu_window(seq[:100], title=label, save_path=fig_dir / f'{label}.png')

if HIFD_RAW_DIR.exists():
    hifd_trials = find_hifd_trials(HIFD_RAW_DIR)
    trial = next((t for t in hifd_trials if t['class_dir'] == 'fall'), None)
    if trial:
        seq = load_hifd_mat(trial['mat_path'], convert_g_to_ms2=config.convert_g_to_ms2)
        if seq is not None and len(seq) >= 100:
            plot_imu_window(seq[:100], title='hifd_fall_raw', save_path=fig_dir / 'hifd_fall_raw.png')

if BITS_RAW_DIR.exists():
    bits_trials = find_bits_trials(BITS_RAW_DIR)
    trial = next((t for t in bits_trials if t['class_dir'] == 'fall'), None)
    if trial:
        seq20 = parse_bits_csv(trial['csv_path'], accel_source=config.bits_accel_source)
        if seq20 is not None:
            seq50 = resample_bits_sequence_to_50hz(seq20, config.bits_original_fs, config.bits_target_fs)
            if seq50 is not None:
                plot_imu_window(seq20[:min(len(seq20), 100)], title='bits_fall_before_resampling_20hz', save_path=fig_dir / 'bits_fall_raw_20hz.png')
                plot_bits_resampling(seq20, seq50, save_path=fig_dir / 'bits_resampling_validation.png')
                print('BITS original shape:', seq20.shape, 'resampled shape:', seq50.shape, 'final model window length:', X.shape[1])

## Processed window examples and dataset statistics

In [ ]:
for direction in ['forward', 'backward', 'lateral']:
    idx = metadata.index[metadata['direction_label'].eq(direction)].tolist()
    if idx:
        plot_imu_window(X[idx[0]], title=f'processed_{direction}_fall_window', save_path=fig_dir / f'processed_{direction}_fall.png')

idx = metadata.index[metadata['fall_label'].eq(0)].tolist()
if idx:
    plot_imu_window(X[idx[0]], title='processed_non_fall_window', save_path=fig_dir / 'processed_non_fall.png')

plot_dataset_distribution(metadata, save_path=fig_dir / 'windows_by_dataset.png')
plot_class_distribution(metadata, 'fall_label', 'Windows by fall label', save_path=fig_dir / 'windows_by_fall_label.png')
plot_class_distribution(metadata, 'direction_label', 'Windows by direction label', save_path=fig_dir / 'windows_by_direction_label.png')
plot_class_distribution(metadata, 'split', 'Windows by split', save_path=fig_dir / 'windows_by_split.png')
plot_channel_stats(X_raw, 'Channel stats before normalization', save_path=fig_dir / 'channel_stats_before_norm.png')
plot_channel_stats(X, 'Channel stats after normalization', save_path=fig_dir / 'channel_stats_after_norm.png')

print('Quality checks')
print('X shape:', X.shape)
print('NaN count:', np.isnan(X).sum())
print('Inf count:', np.isinf(X).sum())
print('\nWindows by dataset:')
print(metadata['dataset'].value_counts())
print('\nSubjects by dataset:')
print(metadata.groupby('dataset')['subject_id'].nunique())
print('\nFall/non-fall ratio:')
print(metadata['fall_label'].value_counts(normalize=True))
print('\nBITS original/resampled length summary:')
display(metadata[metadata['dataset'].eq('bits')][['original_length', 'resampled_length']].describe())